# Evaluate an Italian base language model

This is the main user notebook. Set your Hugging Face model id or Drive checkpoint below, choose a bounded smoke run first, and run the cells in order. The final cell downloads the complete reproducibility bundle.


## 1. Select a GPU runtime

In Colab choose **Runtime → Change runtime type → GPU**. CPU works for tiny tests but is not suitable for most real models.


In [ ]:
# REQUIRED: Hugging Face repo id, or a path under /content/drive after mounting Drive.
MODEL_SOURCE = "your-org/your-italian-base-model"
TOKENIZER_SOURCE = ""  # Empty means: use MODEL_SOURCE.
MODEL_REVISION = None  # Automatically resolve the model repository's current commit SHA.
TOKENIZER_REVISION = None  # Resolve independently when TOKENIZER_SOURCE is another repository.
HF_TOKEN = ""  # Required only for gated/private Hub models.
MOUNT_GOOGLE_DRIVE = False

# Start bounded. Increase limits only after this succeeds.
ENABLE_LIGHTEVAL = False
LIGHTEVAL_SUITE = "verified_windows"
MAX_LIGHTEVAL_SAMPLES = 2
MAX_BLIMP_SAMPLES = 20
MAX_PPL_DOCUMENTS = 3
MAX_PPL_TOKENS_PER_DOCUMENT = 256
MAX_GENERATION_PROMPTS = 3

MODEL_DEVICE = None  # None selects CUDA when available.
MODEL_DTYPE = "auto"
MODEL_BATCH_SIZE = 1


In [ ]:
import os
import shutil
from pathlib import Path

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

%cd /content
repo_path = Path("/content/it_eval_autoregressive_llms")
if repo_path.exists():
    shutil.rmtree(repo_path)
!git clone https://github.com/GiorgosPeikos/it_eval_autoregressive_llms.git
%cd /content/it_eval_autoregressive_llms


In [ ]:
!python -m pip install --upgrade "pip<27" "setuptools<82" wheel
!python -m pip install "lighteval[multilingual]==0.13.0" --no-deps
!python -m pip install -r constraints/lighteval-python310-313.txt
!python -m pip install -e . --no-deps


In [ ]:
import torch

selected_device = MODEL_DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: no GPU detected. Use a Colab GPU runtime for real models.")


In [ ]:
from pathlib import Path
import yaml

config = {
    "run_name": "colab_model_eval",
    "model": {
        "source": MODEL_SOURCE,
        "revision": MODEL_REVISION,
        "tokenizer_source": TOKENIZER_SOURCE or MODEL_SOURCE,
        "tokenizer_revision": TOKENIZER_REVISION,
        "device": selected_device,
        "dtype": MODEL_DTYPE,
        "batch_size": MODEL_BATCH_SIZE,
    },
    "output": {"root_dir": "evaluation_results", "overwrite": True, "save_details": True},
    "runtime": {"seed": 13},
    "lighteval": {
        "enabled": ENABLE_LIGHTEVAL, "suite": LIGHTEVAL_SUITE if ENABLE_LIGHTEVAL else None,
        "max_samples": MAX_LIGHTEVAL_SAMPLES, "dataset_loading_processes": 1,
    },
    "blimp_it": {
        "enabled": True, "max_samples": MAX_BLIMP_SAMPLES,
        "dataset_revision": "4159ecb68388283488cb1d235a7e1946489bc62d",
    },
    "perplexity": {
        "enabled": True, "dataset_repo": "gsarti/clean_mc4_it", "dataset_subset": "tiny",
        "dataset_revision": "167d5696e91ac89f17936f9d0059031cbc4c9e99",
        "dataset_trust_remote_code": True, "dataset_streaming": True, "split": "validation",
        "text_field": "text", "sequence_length": 512, "stride": 256,
        "max_documents": MAX_PPL_DOCUMENTS,
        "max_tokens_per_document": MAX_PPL_TOKENS_PER_DOCUMENT,
    },
    "generation": {
        "enabled": True, "prompts_path": "configs/generation_prompts.yaml",
        "max_prompts": MAX_GENERATION_PROMPTS, "seed": 13,
    },
}
config_path = Path("configs/colab_model_eval.yaml")
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))


## 2. Run the bounded evaluation

The first run intentionally uses small limits. If it succeeds, increase the limits above or enable LightEval and rerun from the configuration cell.


In [ ]:
!python -m it_eval_framework.runners.run_all --config configs/colab_model_eval.yaml


In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

run_config = max(Path("evaluation_results").rglob("run_config.yaml"), key=lambda path: path.stat().st_mtime)
run_dir = run_config.parent
print(f"Complete result directory: {run_dir}")
display(pd.read_csv(run_dir / "summary.csv"))
print((run_dir / "report.md").read_text(encoding="utf-8"))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("italian_model_evaluation", "zip", root_dir=run_dir)
files.download(archive)


## 3. What to keep

Keep the downloaded ZIP. It contains the resolved configuration, environment and revision metadata, raw outputs, normalized metrics, summary, and report. Smoke scores only prove that the pipeline works; use a genuinely held-out Italian corpus and appropriate sample limits before reporting research results.
